# TraderPro — PPO BTC robuste

Ce notebook entraîne un agent PPO **hors ligne** sur des bougies BTC historiques. Il ne contacte pas Binance pendant l'entraînement et ne peut envoyer aucun ordre. Les données doivent être fournies sous forme de CSV local (Kaggle Dataset, Google Drive ou fichier uploadé).

In [ ]:
!pip -q install stable-baselines3[extra] gymnasium pandas numpy matplotlib

import json
import random
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CallbackList, EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


## 1. Charger un CSV local

Le CSV doit contenir `timestamp` (ou `open_time`), `open`, `high`, `low`, `close`, `volume`. Sur Kaggle, renseigne le chemin dans `/kaggle/input/...`. Sur Colab, upload le fichier ou monte Drive, puis change `DATA_PATH`.

In [ ]:
# Mets un chemin CSV existant ici si tu en as déjà un. Sinon le notebook télécharge
# une fois les bougies publiques Binance et les met en cache sous forme de CSV.
DATA_PATH = None
SYMBOL = 'BTCUSDT'
INTERVAL = '1h'
LOOKBACK_DAYS = 730
CACHE_PATH = Path(f'./{SYMBOL}_{INTERVAL}_{LOOKBACK_DAYS}d.csv')

def download_binance_klines(symbol, interval, lookback_days):
    import time
    import requests
    interval_ms = {'1m': 60_000, '5m': 300_000, '15m': 900_000, '1h': 3_600_000, '4h': 14_400_000, '1d': 86_400_000}[interval]
    end_time = int(time.time() * 1000)
    start_time = end_time - lookback_days * 24 * 60 * 60 * 1000
    rows = []
    # data-api.binance.vision est l'endpoint officiel dédié aux données publiques,
    # distinct de l'API de trading souvent bloquée (HTTP 451) dans Kaggle.
    hosts = ['https://data-api.binance.vision', 'https://api.binance.us']
    while start_time < end_time:
        params = {'symbol': symbol, 'interval': interval, 'startTime': start_time, 'endTime': end_time, 'limit': 1000}
        batch, last_error = None, None
        for host in hosts:
            try:
                response = requests.get(f'{host}/api/v3/klines', params=params, timeout=30)
                response.raise_for_status()
                batch = response.json()
                break
            except requests.RequestException as exc:
                last_error = exc
        if batch is None:
            raise RuntimeError(f'Aucun endpoint public disponible : {last_error}')
        if not batch:
            break
        rows.extend(batch)
        start_time = batch[-1][0] + interval_ms
        time.sleep(0.15)
    columns = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume', 'trades', 'taker_base', 'taker_quote', 'ignore']
    result = pd.DataFrame(rows, columns=columns)
    result['timestamp'] = pd.to_datetime(result['open_time'], unit='ms', utc=True)
    return result[['timestamp', 'open', 'high', 'low', 'close', 'volume']]

if DATA_PATH:
    path = Path(DATA_PATH)
elif CACHE_PATH.exists():
    path = CACHE_PATH
    print(f'CSV cache utilisé : {path}')
else:
    print(f'Téléchargement de {LOOKBACK_DAYS} jours de {SYMBOL} en {INTERVAL}…')
    try:
        downloaded = download_binance_klines(SYMBOL, INTERVAL, LOOKBACK_DAYS)
    except Exception as exc:
        raise RuntimeError(
            f'Téléchargement Binance impossible : {exc}. Sur Kaggle, ajoute un Dataset de bougies BTC puis renseigne DATA_PATH.'
        ) from exc
    downloaded.to_csv(CACHE_PATH, index=False)
    path = CACHE_PATH
    print(f'CSV créé : {path} ({len(downloaded):,} bougies)')

raw = pd.read_csv(path)
time_column = 'timestamp' if 'timestamp' in raw.columns else 'open_time'
if time_column not in raw.columns:
    raise ValueError('Le CSV doit contenir timestamp ou open_time.')

if time_column == 'open_time' and np.issubdtype(raw[time_column].dtype, np.number):
    raw['timestamp'] = pd.to_datetime(raw[time_column], unit='ms', utc=True)
else:
    raw['timestamp'] = pd.to_datetime(raw[time_column], utc=True)

required = ['open', 'high', 'low', 'close', 'volume']
missing = set(required) - set(raw.columns)
if missing:
    raise ValueError(f'Colonnes manquantes : {sorted(missing)}')

df = raw.set_index('timestamp')[required].astype(float).sort_index().dropna()
df = df[~df.index.duplicated(keep='last')]
if len(df) < 5_000:
    raise ValueError(f'{len(df)} bougies seulement : utilise au moins 5 000 bougies, idéalement 20 000+.')
print(f'{len(df):,} bougies : {df.index.min()} → {df.index.max()}')


In [ ]:
def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame.copy()
    close = data['close']
    returns = close.pct_change()
    data['ret_1'] = returns.clip(-0.15, 0.15)
    data['ret_4'] = close.pct_change(4).clip(-0.30, 0.30)
    data['ret_24'] = close.pct_change(24).clip(-0.50, 0.50)
    data['vol_24'] = returns.rolling(24).std().clip(0, 0.20)
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    data['rsi'] = ((100 - 100 / (1 + gain / (loss + 1e-12))) - 50) / 50
    fast, slow = close.ewm(span=12, adjust=False).mean(), close.ewm(span=26, adjust=False).mean()
    macd = fast - slow
    data['macd'] = ((macd - macd.ewm(span=9, adjust=False).mean()) / close).clip(-0.05, 0.05)
    ma20, std20 = close.rolling(20).mean(), close.rolling(20).std()
    data['bb'] = ((close - ma20) / (2 * std20 + 1e-12)).clip(-3, 3) / 3
    ma50 = close.rolling(50).mean()
    data['trend'] = ((close / ma50) - 1).clip(-0.30, 0.30)
    data['volume'] = ((data['volume'] / data['volume'].rolling(24).mean()) - 1).clip(-5, 5) / 5
    data['range'] = ((data['high'] - data['low']) / close).clip(0, 0.20)
    return data.replace([np.inf, -np.inf], np.nan).dropna()

features = build_features(df)
FEATURE_COLUMNS = ['ret_1', 'ret_4', 'ret_24', 'vol_24', 'rsi', 'macd', 'bb', 'trend', 'volume', 'range']
print(f'{len(features):,} lignes utilisables, {len(FEATURE_COLUMNS)} features')


## 2. Environnement de trading

Environnement **USDT-M Futures simplifié**, calé sur le mode démo de TraderPro : `0=HOLD`, `1=LONG`, `2=SHORT`. Une action opposée ferme d'abord la position existante ; l'action suivante pourra ouvrir l'autre sens. Le levier est fixé à 1×, les frais sont appliqués à chaque ouverture/fermeture et une liquidation simplifiée est détectée avec le plus haut/bas de la bougie. Ce simulateur ne remplace pas le moteur de marge exact de Binance (maintenance margin, funding réel et frais de liquidation).

In [ ]:
class BTCFuturesEnv(gym.Env):
    """USDT-M Futures isolé, one-way, 1x — modèle volontairement conservateur."""
    metadata = {'render_modes': []}

    def __init__(self, data, episode_length=720, random_start=True, fee=0.0005, leverage=1, initial_cash=10_000, funding_rate_8h=0.0):
        super().__init__()
        self.data = data.reset_index(drop=True)
        self.episode_length = min(episode_length, len(self.data) - 2)
        self.random_start = random_start
        self.fee, self.leverage = fee, leverage
        self.initial_cash, self.funding_rate_8h = initial_cash, funding_rate_8h
        self.action_space = spaces.Discrete(3)  # HOLD, LONG, SHORT
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(len(FEATURE_COLUMNS) + 2,), dtype=np.float32)

    def _equity(self, price):
        return self.wallet + self.quantity * (price - self.entry_price)

    def _obs(self):
        row = self.data.loc[self.step_index, FEATURE_COLUMNS].to_numpy(dtype=np.float32)
        price = float(self.data.loc[self.step_index, 'close'])
        equity = max(self._equity(price), 1e-12)
        effective_leverage = abs(self.quantity * price) / equity if self.quantity else 0.0
        return np.nan_to_num(np.concatenate([row, [float(self.position), effective_leverage]])).astype(np.float32)

    def _open(self, direction, price):
        notional = max(self._equity(price), 0.0) * self.leverage
        self.position = direction
        self.quantity = direction * notional / price
        self.entry_price = price
        self.wallet -= abs(self.quantity * price) * self.fee
        self.trade_count += 1

    def _close(self, price):
        self.wallet += self.quantity * (price - self.entry_price)
        self.wallet -= abs(self.quantity * price) * self.fee
        self.position, self.quantity, self.entry_price = 0, 0.0, price
        self.trade_count += 1

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        max_start = len(self.data) - self.episode_length - 2
        self.start = int(self.np_random.integers(0, max_start + 1)) if self.random_start else 0
        self.end, self.step_index = self.start + self.episode_length, self.start
        self.wallet, self.position, self.quantity = self.initial_cash, 0, 0.0
        self.entry_price, self.peak, self.trade_count = float(self.data.loc[self.start, 'close']), self.initial_cash, 0
        self.equity_curve = [self.initial_cash]
        return self._obs(), {}

    def step(self, action):
        price = float(self.data.loc[self.step_index, 'close'])
        equity_before = max(self._equity(price), 1e-12)
        desired = {0: 0, 1: 1, 2: -1}[int(action)]
        invalid = False
        if desired and self.position == 0:
            self._open(desired, price)
        elif desired and self.position == -desired:
            self._close(price)  # même règle que l'API TraderPro : fermeture d'abord
        elif desired and self.position == desired:
            invalid = True

        self.step_index += 1
        next_price = float(self.data.loc[self.step_index, 'close'])
        if self.position and self.step_index % 8 == 0 and self.funding_rate_8h:
            self.wallet -= self.position * abs(self.quantity * next_price) * self.funding_rate_8h

        low, high = float(self.data.loc[self.step_index, 'low']), float(self.data.loc[self.step_index, 'high'])
        liquidation_price = np.inf
        if self.quantity > 0:
            liquidation_price = self.entry_price - self.wallet / self.quantity
        elif self.quantity < 0:
            liquidation_price = self.entry_price + self.wallet / abs(self.quantity)
        liquidated = (self.quantity > 0 and low <= liquidation_price) or (self.quantity < 0 and high >= liquidation_price)
        equity_after = 0.0 if liquidated else max(self._equity(next_price), 0.0)
        if liquidated:
            self.wallet, self.position, self.quantity = 0.0, 0, 0.0

        self.peak = max(self.peak, equity_after)
        drawdown = 1 - equity_after / max(self.peak, 1e-12)
        reward = -10.0 if liquidated else float(np.log(max(equity_after, 1e-12) / equity_before) - 0.05 * drawdown - (0.0002 if invalid else 0))
        self.equity_curve.append(equity_after)
        terminated = self.step_index >= self.end or liquidated
        info = {'equity': equity_after, 'drawdown': drawdown, 'trades': self.trade_count, 'position': self.position, 'liquidated': liquidated}
        return self._obs(), reward, terminated, False, info


In [ ]:
# Split strictement chronologique : aucune fuite du futur.
n = len(features)
train = features.iloc[:int(n * 0.70)].copy()
validation = features.iloc[int(n * 0.70):int(n * 0.85)].copy()
test = features.iloc[int(n * 0.85):].copy()
for name, part in [('train', train), ('validation', validation), ('test', test)]:
    print(f'{name:10}: {len(part):,} — {part.index.min()} → {part.index.max()}')

EPISODE_LENGTH = 720  # 30 jours en bougies 1h ; réduit les évaluations interminables
FUTURES_LEVERAGE = 1  # doit rester aligné sur backend/.env
def make_train_env():
    return Monitor(BTCFuturesEnv(train, episode_length=EPISODE_LENGTH, random_start=True, leverage=FUTURES_LEVERAGE))
def make_eval_env():
    return Monitor(BTCFuturesEnv(validation, episode_length=EPISODE_LENGTH, random_start=False, leverage=FUTURES_LEVERAGE))

train_env = VecNormalize(DummyVecEnv([make_train_env]), norm_obs=True, norm_reward=True, clip_obs=10.0)
eval_env = VecNormalize(DummyVecEnv([make_eval_env]), training=False, norm_obs=True, norm_reward=False, clip_obs=10.0)
eval_env.obs_rms = train_env.obs_rms


In [ ]:
class HeartbeatCallback(BaseCallback):
    def __init__(self, every=10_000):
        super().__init__()
        self.every = every
    def _on_step(self):
        if self.num_timesteps % self.every == 0:
            print(f'PPO actif : {self.num_timesteps:,} pas')
        return True

OUTPUT_DIR = Path('traderpro_rl_output')
OUTPUT_DIR.mkdir(exist_ok=True)
eval_callback = EvalCallback(
    eval_env, best_model_save_path=str(OUTPUT_DIR), log_path=str(OUTPUT_DIR),
    eval_freq=20_000, n_eval_episodes=2, deterministic=True, render=False, verbose=1,
)

model = PPO(
    'MlpPolicy', train_env, learning_rate=3e-4, n_steps=2048, batch_size=256,
    n_epochs=10, gamma=0.995, gae_lambda=0.95, clip_range=0.2,
    ent_coef=0.002, policy_kwargs=dict(net_arch=[128, 128]), seed=SEED, verbose=0,
)

# Commence par 100k ; augmente seulement après un backtest correct.
TOTAL_TIMESTEPS = 100_000
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=CallbackList([eval_callback, HeartbeatCallback()]), progress_bar=False)
model.save(str(OUTPUT_DIR / 'ppo_btc_final'))
train_env.save(str(OUTPUT_DIR / 'vecnormalize.pkl'))
print('Entraînement terminé.')


In [ ]:
def backtest(model, data):
    env = BTCFuturesEnv(data, episode_length=len(data) - 2, random_start=False, leverage=FUTURES_LEVERAGE)
    obs, _ = env.reset(seed=SEED)
    done = False
    while not done:
        # VecNormalize est utilisé uniquement pour normaliser l'observation.
        normalized = (obs - train_env.obs_rms.mean) / np.sqrt(train_env.obs_rms.var + 1e-8)
        action, _ = model.predict(np.clip(normalized, -10, 10), deterministic=True)
        obs, _, terminated, truncated, info = env.step(int(action))
        done = terminated or truncated
    curve = np.array(env.equity_curve)
    returns = np.diff(curve) / curve[:-1]
    sharpe = (returns.mean() / (returns.std() + 1e-12)) * np.sqrt(24 * 365)
    drawdown = 1 - curve / np.maximum.accumulate(curve)
    return {'return': curve[-1] / curve[0] - 1, 'sharpe': sharpe, 'max_drawdown': drawdown.max(), 'trades': env.trade_count, 'curve': curve}

best_path = OUTPUT_DIR / 'best_model.zip'
best_model = PPO.load(str(best_path if best_path.exists() else OUTPUT_DIR / 'ppo_btc_final'), env=train_env)
metrics = backtest(best_model, test)
print({key: round(value, 4) for key, value in metrics.items() if key != 'curve'})
plt.figure(figsize=(12, 4))
plt.plot(metrics['curve']); plt.title('Equity curve — test hors-échantillon'); plt.grid(); plt.show()

manifest = {
    'model': 'PPO BTC USDT-M Futures (one-way, isolated)', 'timeframe': 'from CSV', 'features': FEATURE_COLUMNS,
    'fee': 0.0005, 'leverage': FUTURES_LEVERAGE, 'episode_length': EPISODE_LENGTH, 'train_rows': len(train),
    'validation_rows': len(validation), 'test_rows': len(test),
    'test_metrics': {key: float(value) for key, value in metrics.items() if key != 'curve'}
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(f'Fichiers à récupérer : {OUTPUT_DIR}/best_model.zip (si présent), ppo_btc_final.zip, vecnormalize.pkl, manifest.json')


## Avant tout déploiement

Ne charge pas le modèle dans le bot si le test hors-échantillon est inférieur à une stratégie *buy-and-hold* après frais, si le drawdown est excessif, ou si les résultats changent fortement selon la période. Valide ensuite plusieurs semaines sur Binance Testnet avant toute autre étape.